In [1]:
!echo $DISPLAY
%env DISPLAY=localhost:10.0
!echo $DISPLAY


env: DISPLAY=localhost:10.0
localhost:10.0


In [2]:
UNI_RANDOM_SEED = 2024

import numpy as np
import torch
import torch.nn.functional as F

np.random.seed(UNI_RANDOM_SEED) 
torch.manual_seed(UNI_RANDOM_SEED)

torch.cuda.manual_seed(UNI_RANDOM_SEED)
torch.cuda.manual_seed_all(UNI_RANDOM_SEED)

import pdb
from pathlib import Path

try:
    import open3d
    from visual_utils import open3d_vis_utils as V
    OPEN3D_FLAG = True
except:
    import mayavi.mlab as mlab
    from visual_utils import visualize_utils as V
    OPEN3D_FLAG = False

from pcdet.datasets.kitti.kitti_dataset import create_kitti_infos
from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.datasets import KittiDataset, build_dataloader
from pcdet.models import build_network, load_data_to_gpu
from pcdet.utils import common_utils

from eval_utils import eval_utils


EVAL_OUTPUT_DIR = "./eval_output/"
CFG_FILE = "./cfgs/kitti_models/pointrcnn.yaml"
DATA_CONFIG_FILE = "./cfgs/dataset_configs/kitti_dataset.yaml"
DATA_PATH = "/home/ksas/Public/datasets/KITTI"
CKPT_PATH = "/home/ksas/Public/model_zoo/pcdet/pointrcnn_7870.pth"

BATCH_SIZE = 1
WORKERS = 4
DIST_TEST = False

cfg_from_yaml_file(CFG_FILE, cfg)

# BATCH_SIZE = cfg.OPTIMIZATION.BATCH_SIZE_PER_GPU
logger = common_utils.create_logger()
logger.info('-----------------Gradient Fetching Test-------------------------')

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[Open3D INFO] Resetting default logger to print to terminal.


/home/ksas/chw_space/OpenPCDet_Developing/pcdet/models/detectors/__init__.py:20: UserWarning: You are using a variant OpenPCDet modified by uzuki-dev, NOT AN ORIGINAL VERSION!
  warnings.warn("You are using a variant OpenPCDet modified by uzuki-dev, NOT AN ORIGINAL VERSION!")
2024-01-10 15:41:20,158   INFO  -----------------Gradient Fetching Test-------------------------


In [3]:
test_set, test_loader, sampler = build_dataloader(
        dataset_cfg=cfg.DATA_CONFIG,
        class_names=cfg.CLASS_NAMES,
        batch_size=BATCH_SIZE,
        dist=DIST_TEST, workers=WORKERS, logger=logger, training=False
    )
logger.info(f'Class names of samples: \t{test_set.class_names}')

2024-01-10 15:41:20,176   INFO  Loading KITTI dataset
2024-01-10 15:41:20,312   INFO  Total samples for KITTI dataset: 3769
2024-01-10 15:41:20,314   INFO  Class names of samples: 	['Car', 'Pedestrian', 'Cyclist']


In [4]:
model = build_network(model_cfg=cfg.MODEL, num_class=len(cfg.CLASS_NAMES), dataset=test_set)
model.load_params_from_file(filename=CKPT_PATH, logger=logger, to_cpu=True)
model.cuda()
model.eval()

for idx, module in enumerate(model.module_list):
    logger.info(f'Module names of model \t({idx}): \t{module._get_name()}')
    
backbone_network = model.module_list[0]
point_headbox = model.module_list[1]
pointrcnn_head = model.module_list[2]

2024-01-10 15:41:21,312   INFO  ==> Loading parameters from checkpoint /home/ksas/Public/model_zoo/pcdet/pointrcnn_7870.pth to CPU
2024-01-10 15:41:21,460   INFO  ==> Done (loaded 309/309)
2024-01-10 15:41:21,473   INFO  Module names of model 	(0): 	PointNet2MSG
2024-01-10 15:41:21,475   INFO  Module names of model 	(1): 	PointHeadBox
2024-01-10 15:41:21,482   INFO  Module names of model 	(2): 	PointRCNNHead


In [5]:

def pseudo_train_test():
    for i, batch_dict in enumerate(test_loader):
        load_data_to_gpu(batch_dict)
        logger.info(f"keys of batch dict: \t{batch_dict.keys()}")
        
        model.eval()
        model.pseudo_train()
        model.zero_grad()
        pred_dicts, _ = model(batch_dict)
        # pred_dicts, _, _ = model(batch_dict)

        
        loss, tb_dict, disp_dict = model.get_training_loss()
        logger.info(f"total loss: \t{loss}")
        
        loss_dict = {}
       
        point_headbox_cls_loss, cls_loss_dict = point_headbox.get_cls_layer_loss()
        point_headbox_box_loss, box_loss_dict = point_headbox.get_box_layer_loss()
        loss_dict.update(cls_loss_dict)
        loss_dict.update(box_loss_dict)
        
        rcnn_cls_loss, cls_loss_dict = pointrcnn_head.get_box_cls_layer_loss()
        rcnn_reg_loss, reg_loss_dict = pointrcnn_head.get_box_reg_layer_loss()
        loss_dict.update(cls_loss_dict)
        loss_dict.update(reg_loss_dict)
        
        logger.info(f"loss dict: \t{loss_dict}")
        
        V.draw_scenes(
            points=batch_dict['points'][:, 1:], ref_boxes=pred_dicts[0]['pred_boxes'].detach(),
            ref_scores=pred_dicts[0]['pred_scores'].detach(), ref_labels=pred_dicts[0]['pred_labels'].detach(), gt_boxes=batch_dict['gt_boxes'][0]
        )
        
pseudo_train_test()

2024-01-10 15:41:22,757   INFO  keys of batch dict: 	dict_keys(['frame_id', 'calib', 'gt_boxes', 'points', 'lidar_aug_matrix', 'use_lead_xyz', 'image_shape', 'batch_size'])
2024-01-10 15:41:24,489   INFO  total loss: 	1.7824331521987915
2024-01-10 15:41:24,503   INFO  loss dict: 	{'point_loss_cls': 0.2677958607673645, 'point_pos_num': 27.0, 'point_loss_box': 1.0907220840454102, 'rcnn_loss_cls': 0.03253515064716339, 'rcnn_loss_reg': 0.33158567547798157, 'rcnn_loss_corner': 0.05979444831609726}


[Open3D WARNING] invalid color in PaintUniformColor, clipping to [0, 1]
[Open3D WARNING] invalid color in PaintUniformColor, clipping to [0, 1]


2024-01-10 15:41:36,800   INFO  keys of batch dict: 	dict_keys(['frame_id', 'calib', 'gt_boxes', 'points', 'lidar_aug_matrix', 'use_lead_xyz', 'image_shape', 'batch_size'])
2024-01-10 15:41:36,865   INFO  total loss: 	0.7812682390213013
2024-01-10 15:41:36,870   INFO  loss dict: 	{'point_loss_cls': 0.36322546005249023, 'point_pos_num': 56.0, 'point_loss_box': 0.23015902936458588, 'rcnn_loss_cls': 0.007299348246306181, 'rcnn_loss_reg': 0.11559773981571198, 'rcnn_loss_corner': 0.06498667597770691}
2024-01-10 15:43:09,879   INFO  keys of batch dict: 	dict_keys(['frame_id', 'calib', 'gt_boxes', 'points', 'lidar_aug_matrix', 'use_lead_xyz', 'image_shape', 'batch_size'])
2024-01-10 15:43:09,939   INFO  total loss: 	0.3781856298446655
2024-01-10 15:43:09,945   INFO  loss dict: 	{'point_loss_cls': 0.06093010678887367, 'point_pos_num': 99.0, 'point_loss_box': 0.16048863530158997, 'rcnn_loss_cls': 0.021521365270018578, 'rcnn_loss_reg': 0.10196714103221893, 'rcnn_loss_corner': 0.03327837213873863

In [ ]:
# eval_utils.eval_one_epoch(
#         cfg, None, model, test_loader, 0, logger, dist_test=DIST_TEST,
#         result_dir=None # Path(EVAL_OUTPUT_DIR)
#         , infer_time=True
#     )